# Validation Tests for creating dimensional tables in gold layer


![Logical Model](../../dimensional_modelling/assets/logical_model.png)

![Physical Model](../../dimensional_modelling/assets/physical_model.png)

## 1. `dim_event`

In [0]:
%sql
-- number of unique events
SELECT COUNT(*) AS dim_event_row_count
FROM marathos_catalog.gold.dim_event;

In [0]:
%sql
-- check the unique values and check for duplicates
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT event_id) AS distinct_event_ids,
    COUNT(*) - COUNT(DISTINCT event_id) AS duplicate_event_ids
FROM marathos_catalog.gold.dim_event;

In [0]:
%sql
-- display
SELECT *
FROM marathos_catalog.gold.dim_event
ORDER BY event_name
LIMIT 100;

## 2. `dim_athlete`

In [0]:
%sql
-- number of unique athletes
SELECT COUNT(*) AS dim_athlete_row_count
FROM marathos_catalog.gold.dim_athlete;

In [0]:
%sql
-- check the unique values and check for duplicates
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT athlete_id) AS distinct_athlete_ids,
    COUNT(*) - COUNT(DISTINCT athlete_id) AS duplicate_athlete_ids
FROM marathos_catalog.gold.dim_athlete;

In [0]:
%sql
-- display
SELECT *
FROM marathos_catalog.gold.dim_athlete
ORDER BY athlete_id
LIMIT 100;

## 3. `dim_country`

In [0]:
%sql
-- number of unique countries
SELECT COUNT(*) AS dim_country_row_count
FROM marathos_catalog.gold.dim_country;

In [0]:
%sql
-- check the unique values and check for duplicates
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT country_code_iso3) AS distinct_country_codes,
    COUNT(*) - COUNT(DISTINCT country_code_iso3) AS duplicate_country_codes
FROM marathos_catalog.gold.dim_country;

In [0]:
%sql
-- display
SELECT *
FROM marathos_catalog.gold.dim_country
ORDER BY country_name
LIMIT 100;

## 4- `dim_date`

In [0]:
%sql
-- number of unique event dates
SELECT COUNT(*) AS dim_date_row_count
FROM marathos_catalog.gold.dim_date;

In [0]:
%sql
-- check the unique values and check for duplicates
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT date_id) AS distinct_date_ids,
    COUNT(*) - COUNT(DISTINCT date_id) AS duplicate_date_ids
FROM marathos_catalog.gold.dim_date;

In [0]:
%sql
-- preview date dimension
SELECT *
FROM marathos_catalog.gold.dim_date
ORDER BY event_start_date
LIMIT 20;

## 5. `fact_results`

In [0]:
%sql
-- check the unique values and check for duplicates
SELECT COUNT(*) AS fact_results_row_count
FROM marathos_catalog.gold.fact_results;

In [0]:
%sql
-- check the unique values and check for duplicates
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT result_id) AS distinct_result_ids,
    COUNT(*) - COUNT(DISTINCT result_id) AS duplicate_result_ids
FROM marathos_catalog.gold.fact_results;

In [0]:
%sql
-- display
SELECT *
FROM marathos_catalog.gold.fact_results
LIMIT 100;

# Validate fact to dimension relationships
## 1. `Fact` → `Event`

In [0]:
%sql
SELECT COUNT(*) AS missing_event_keys
FROM marathos_catalog.gold.fact_results f
LEFT JOIN marathos_catalog.gold.dim_event e
    ON f.event_id = e.event_id
WHERE e.event_id IS NULL;

## 2. `Fact` → `Athlete`

In [0]:
%sql
SELECT COUNT(*) AS missing_athlete_keys
FROM marathos_catalog.gold.fact_results f
LEFT JOIN marathos_catalog.gold.dim_athlete a
    ON f.athlete_id = a.athlete_id
WHERE a.athlete_id IS NULL;

## 3. `Fact` → `Country`

In [0]:
%sql
SELECT COUNT(*) AS missing_country_keys
FROM marathos_catalog.gold.fact_results f
LEFT JOIN marathos_catalog.gold.dim_country c
    ON f.country_code_iso3 = c.country_code_iso3
WHERE c.country_code_iso3 IS NULL;

## 4. `Fact` → `Date`

In [0]:
%sql
SELECT COUNT(*) AS missing_date_keys
FROM marathos_catalog.gold.fact_results f
LEFT JOIN marathos_catalog.gold.dim_date d
    ON f.date_id = d.date_id
WHERE d.date_id IS NULL;

In [0]:
%sql
-- display
SELECT *
FROM marathos_catalog.gold.dim_date
ORDER BY event_start_date
LIMIT 100;

#### Findings:
- dim_event: passed
- dim_athlete: passed
- dim_country: passed
- dim_date: passed
- fact_results: passed
- fact-to-dimension relationship checks: all 0 missing keys

## Star Schema Full Preview

In [0]:
%sql
SELECT
    f.result_id,
    d.event_start_date,
    d.year,
    d.month_name,
    e.event_name,
    e.event_distance_length,
    e.event_distance_type,
    a.athlete_id,
    a.athlete_gender,
    a.athlete_year_of_birth,
    a.athlete_age_category,
    c.country_name,
    f.athlete_performance,
    f.athlete_performance_type,
    f.athlete_average_speed_kmh,
    f.athlete_age_at_event
FROM marathos_catalog.gold.fact_results f
LEFT JOIN marathos_catalog.gold.dim_event e
    ON f.event_id = e.event_id
LEFT JOIN marathos_catalog.gold.dim_athlete a
    ON f.athlete_id = a.athlete_id
LEFT JOIN marathos_catalog.gold.dim_country c
    ON f.country_code_iso3 = c.country_code_iso3
LEFT JOIN marathos_catalog.gold.dim_date d
    ON f.date_id = d.date_id
LIMIT 100;